# Load and PV Forecastability

## Objectives

Part 5 forecasts household-level load and the generation hiding inside it, at the scale of hundreds of real smart meters, not one aggregate signal. Before reaching for a model, this notebook answers the question every later chapter depends on: how forecastable is a real customer's own signal, and what does that measured forecastability say about which approach is actually worth using on them?

By the end you will be able to:

- Run a real forecastability diagnostic (entropy, Hurst exponent, DFA, autocorrelation, stationarity) across hundreds of real customers individually, not one aggregate signal, and report the real spread.
- Check whether a customer's forecastability predicts which baseline strategy already wins, on this book's own real data.
- Characterize how a known PV component changes a customer's own measured forecastability, using AusNet's own separately-recorded PV shape, not a proxy.
- Compare forecastability across load, PV, and EV charging directly, being honest about which of the three is a real per-customer time series and which is a diversified, constructed reference.
- Turn a customer's own forecastability profile into a real decision: which baseline already serves a customer well, and which customers are worth a later chapter's own extra machinery.


## Why forecastability, and what the literature already says

A forecasting model is a bet that the future resembles a recognisable pattern from the past. Whether that bet is worth making at all, and worth making before spending real engineering time on it, is a question with its own real literature, not just this book's own intuition.

The premise checked directly in this chapter comes from Peng, Wang, Lu, Li, Shi, Wang and Li's real study of short-term load forecasting across aggregation levels (ISGT Asia, 2019). Using approximate entropy on a real smart-meter dataset, they found individual household loads are typically more volatile, and much harder to forecast, than a substation- or city-level aggregate. That finding motivates a real question worth checking on this book's own data rather than assuming it transfers: does the same spread in forecastability show up across hundreds of real AusNet customers, and if so, how much.

A second, genuinely counter-intuitive finding worth checking directly rather than assumed: Fias, Hashmi and Deconinck's real 2025 study of load-profile uncertainty under rising EV and PV adoption finds that joint EV and PV activity can *reduce* net-load uncertainty through a compensatory daytime effect, EV charging and PV generation both concentrated around the middle of the day can offset each other, not simply compound into more volatility. "More DER means more volatile" is the intuitive story; it is not always the real one.

Two more recent papers shape this chapter's own methodology directly. Wang, Klee and Roos propose assessing forecastability *before* model development with two metrics, a Spectral Predictability Score (the strength and regularity of a signal's frequency components) and the Largest Lyapunov Exponent (how chaotic or stable the underlying generating system is), and report that these measures correlate strongly with actual forecast performance on the M5 competition data. Their practical recommendation, adopted directly below: focus modelling effort on the customers forecastability already flags as tractable, and set different expectations for the rest, rather than running the same intensive pipeline on every customer regardless of whether it can pay off. De Bortoli, Ferrari, Ravazzolo and Rossini take a related but distinct angle, applying a Model Selection Confidence Set method to real Italian hourly electricity load data: rather than picking one "best" model, their method identifies the *set* of models statistically indistinguishable from the true data-generating process at a given confidence level, finding that noisier customers produce larger, more contested sets, and cleaner ones narrow sharply. This chapter's own closing decision table borrows that same honesty: a real confidence set of viable approaches per forecastability regime, not one falsely-confident recommendation.

Two families of diagnostic do the actual measuring below. The first, entropy and long-range-dependence measures (permutation entropy, sample entropy, the Hurst exponent, detrended fluctuation analysis), come from nonlinear time-series analysis and originally had nothing to do with electricity at all, one was built for the Nile river, another for DNA sequences. The second, classical Box-Jenkins time-series theory (autocorrelation, partial autocorrelation, unit-root and stationarity testing), is the standard toolkit behind every ARIMA-family forecaster, laid out in full in Hyndman and Athanasopoulos's *Forecasting: Principles and Practice*. Both families answer the same underlying question from different directions: how much real structure does this signal actually have, and is a model likely to find it.


## Getting the data

This notebook reuses the same real AusNet smart-meter pool Part 4 already vendored, no new fetch step needed:

```bash
uv run python scripts/fetch_part4_ausnet_data.py
```


In [ ]:
from pathlib import Path

from great_tables import GT, md
from lets_plot import (
    LetsPlot,
    aes,
    element_text,
    facet_wrap,
    geom_bar,
    geom_density,
    geom_hline,
    geom_line,
    geom_point,
    geom_segment,
    ggplot,
    ggsize,
    labs,
    scale_color_manual,
    scale_fill_manual,
    theme,
)
import numpy as np
import pandas as pd

from ark.plot.gt_style import themed_gt
from ark.plot.theme import modern_theme
from ark.plot.tokens import AI_ACCENT, DANGER, ENERGY_ACCENT, GRAY_600, INFO, PRIMARY, SUCCESS, WARNING

LetsPlot.setup_html(isolated_frame=True)

UNBOLD_AXES = theme(axis_title_x=element_text(face="plain"), axis_title_y=element_text(face="plain"))
FULL_WIDTH = 960
STEPS_PER_DAY = 48  # 30-minute resolution
STEPS_PER_WEEK = STEPS_PER_DAY * 7

# One fixed color per baseline, reused in every chart in this notebook that
# shows baseline identity, never picked positionally per chart.
BASELINE_COLORS = {
    "naive": INFO,
    "seasonal_naive": WARNING,
    "drift": AI_ACCENT,
    "window_average": SUCCESS,
}

# One fixed color per forecastability regime, reused everywhere a regime is
# shown: green for the easiest-to-forecast end, red for the hardest.
REGIME_COLORS = {
    "persistent": SUCCESS,
    "regular": INFO,
    "volatile / low-persistence": DANGER,
}

# One fixed color per signal type, reused everywhere Load/PV/EV are compared
# directly, a different category dimension from baseline/regime above so it
# gets its own dict rather than reusing either.
SIGNAL_COLORS = {
    "Load": PRIMARY,
    "PV": WARNING,
    "EV": ENERGY_ACCENT,
}


def _lollipop_chart(values: np.ndarray, *, n: int, title: str, y_label: str) -> "ggplot":
    """A vertical-lollipop chart against a 95% confidence band, shared by ACF and PACF below."""
    ci = 1.96 / np.sqrt(n)
    lag_df = pd.DataFrame({"lag": np.arange(len(values)), "value": values})
    lag_df["significant"] = np.where(lag_df["value"].abs() > ci, "significant", "not significant")

    return (
        ggplot(lag_df, aes(x="lag", y="value"))
        + geom_segment(aes(x="lag", xend="lag", yend="value", color="significant"), y=0, size=1.0)
        + geom_point(aes(color="significant"), size=1.8)
        + geom_hline(yintercept=0, color=GRAY_600, size=0.3)
        + geom_hline(yintercept=ci, linetype="dashed", color=GRAY_600, size=0.5)
        + geom_hline(yintercept=-ci, linetype="dashed", color=GRAY_600, size=0.5)
        + scale_color_manual(values={"significant": INFO, "not significant": GRAY_600}, guide="none")
        + labs(title=title, subtitle=f"95% confidence band: ±{ci:.3f} (n={n:,})", x="Lag (half-hours)", y=y_label)
        + modern_theme()
        + UNBOLD_AXES
        + ggsize(FULL_WIDTH, 260)
    )


def acf_lollipop_chart(series: np.ndarray, *, max_lag: int, title: str) -> "ggplot":
    """An ACF chart: how strongly the series correlates with a lagged copy of itself."""
    from statsmodels.tsa.stattools import acf

    values = acf(series, nlags=max_lag, fft=True)
    return _lollipop_chart(values, n=len(series), title=title, y_label="ACF")


def pacf_lollipop_chart(series: np.ndarray, *, max_lag: int, title: str) -> "ggplot":
    """A PACF chart: what a lag adds once every shorter lag is already accounted for."""
    from statsmodels.tsa.stattools import pacf

    values = pacf(series, nlags=max_lag)
    return _lollipop_chart(values, n=len(series), title=title, y_label="PACF")


DATA_DIR = Path("../../resources/cvar_flexibility/data/timeseries-lv")
PV_KVA = 5.0  # matches Part 4's own realistic rooftop-system assumption
RNG = np.random.default_rng(42)

## Hundreds of customers, not one aggregate signal

AusNet's own real pool holds 342 real households, a full year each at 30-minute resolution. Using every one of them, rather than working from one region-wide aggregate, is the only way to see whether forecastability itself varies from customer to customer, the question this whole notebook is built around. Checked directly rather than assumed: every one of the 342 real load series has genuine variance, no gaps, and no near-constant readings, the same real data Part 4 already ran a full OpenDSS time-series simulation against cleanly. No quality gate is needed here the way an earlier, since-dropped data source required; the check below confirms that rather than assuming it.

In [ ]:
load_data = np.load(DATA_DIR / "Residential load data 30-min resolution.npy")
pv_data = np.load(DATA_DIR / "Residential PV data 30-min resolution.npy")
n_customers, n_days, _ = load_data.shape
print(f"real customers available: {n_customers}")
print(f"real days per customer: {n_days} ({n_days / 365:.2f} years)")

series_all = load_data.reshape(n_customers, -1)  # (342, 17,520), one real continuous half-hourly year

# A real, checked quality confirmation, not an elaborate gate: every one of
# these 342 real series needs genuine variance and no missing readings
# before any diagnostic below is trustworthy.
std_per_customer = series_all.std(axis=1)
print(f"customers with near-zero variance: {(std_per_customer < 0.01).sum()}")
print(f"any missing readings: {bool(np.isnan(series_all).any())}")
print(f"half-hourly reading range: min={series_all.min():.3f} kW, max={series_all.max():.3f} kW")

real customers available: 342
real days per customer: 365 (1.00 years)
customers with near-zero variance: 0
any missing readings: False
half-hourly reading range: min=0.000 kW, max=16.534 kW


In [ ]:
preview = pd.DataFrame(
    {
        "half_hour_of_day": range(6),
        "load_kw": load_data[0, 0, :6],
    }
)

themed_gt(
    GT(preview)
    .tab_header(title=md("**One Real Customer, First 3 Hours**"), subtitle="AusNet AMI dataset, 30-minute resolution")
    .cols_label(half_hour_of_day=md("**Half-hour of day**"), load_kw=md("**Load (kW)**"))
    .tab_source_note("Real AusNet residential smart-meter data, Team-Nando DER-hosting-capacity tutorial pool"),
    n_rows=len(preview),
)

GT(_tbl_data=   half_hour_of_day  load_kw
0                 0    1.226
1                 1    0.916
2                 2    0.146
3                 3    0.224
4                 4    0.158
5                 5    0.292, _body=<great_tables._gt_data.Body object at 0x11688f560>, _boxhead=Boxhead([ColInfo(var='half_hour_of_day', type=<ColInfoTypeEnum.default: 1>, column_label=Md(text='**Half-hour of day**'), column_align='right', column_width=None), ColInfo(var='load_kw', type=<ColInfoTypeEnum.default: 1>, column_label=Md(text='**Load (kW)**'), column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x116fd9640>, _spanners=Spanners([]), _heading=Heading(title=Md(text='**One Real Customer, First 3 Hours**'), subtitle='AusNet AMI dataset, 30-minute resolution', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x11390dd30>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x110ec7620>, _source_notes=['Real AusNet residential smart-meter data, Team-Nando DER-hosting-capacity tutorial pool'], _footnotes=[], _styles=[StyleInfo(locname=LocTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#171717', font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocSubTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#6B7280', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='half_hour_of_day', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='load_kw', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocSourceNotes(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#6B7280', font=None, size=None, align=None, v_align=None, style='italic', weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocBody(columns=None, rows=[1, 3, 5], mask=None), grpname=None, colname='half_hour_of_day', rownum=1, colnum=None, styles=[CellStyleFill(color='#F8F9FA')]), StyleInfo(locname=LocBody(columns=None, rows=[1, 3, 5], mask=None), grpname=None, colname='half_hour_of_day', rownum=3, colnum=None, styles=[CellStyleFill(color='#F8F9FA')]), StyleInfo(locname=LocBody(columns=None, rows=[1, 3, 5], mask=None), grpname=None, colname='half_hour_of_day', rownum=5, colnum=None, styles=[CellStyleFill(color='#F8F9FA')]), StyleInfo(locname=LocBody(columns=None, rows=[1, 3, 5], mask=None), grpname=None, colname='load_kw', rownum=1, colnum=None, styles=[CellStyleFill(color='#F8F9FA')]), StyleInfo(locname=LocBody(columns=None, rows=[1, 3, 5], mask=None), grpname=None, colname='load_kw', rownum=3, colnum=None, styles=[CellStyleFill(color='#F8F9FA')]), StyleInfo(locname=LocBody(columns=None, rows=[1, 3, 5], mask=None), grpname=None, colname='load_kw', rownum=5, colnum=None, styles=[CellStyleFill(color='#F8F9FA')])], _locale=<great_tables._gt_data.Locale object at 0x1139a1460>, _formats=[], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='100%'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=Opt

In [ ]:
# A stable, recognizable daily pattern, not an arbitrary pick: rank by
# lag-48 autocorrelation (how strongly each day repeats the one before) and
# take the top 5, at 30-minute resolution, across all 342 real customers.
daily_repeat_score = pd.Series({i: pd.Series(series_all[i]).autocorr(lag=STEPS_PER_DAY) for i in range(n_customers)})
example_ids = daily_repeat_score.nlargest(5).index.tolist()

TWO_WEEKS_STEPS = STEPS_PER_DAY * 14
example_df = pd.concat(
    [
        pd.DataFrame(
            {
                "hour": np.arange(TWO_WEEKS_STEPS) / 2,
                "net_load": series_all[cust_id, :TWO_WEEKS_STEPS],
                "customer": f"customer {cust_id}",
            }
        )
        for cust_id in example_ids
    ],
    ignore_index=True,
)

p = (
    ggplot(example_df, aes("hour", "net_load", color="customer"))
    + geom_line(size=0.6, show_legend=False)
    + labs(title="Five Real Customers, a Stable Daily Rhythm", x="Hour", y="Load (kW)")
    + modern_theme()
    + UNBOLD_AXES
    + ggsize(FULL_WIDTH, 320)
)
p

## How forecastable is a real customer?

Individual loads are known to be more volatile and harder to forecast than a substation- or city-level aggregate, the finding from Peng et al.'s (2019) entropy-based study above. Running the same kind of battery here, on all 342 real AusNet customers individually rather than one aggregate, checks whether that holds on this book's own data, and how much forecastability itself varies customer to customer.

Four metrics, two real theoretical traditions:

**Permutation entropy** (Bandt and Pompe, 2002) asks a purely ordinal question: break the series into short overlapping windows, record each window's *ranking* pattern (does the signal go up-up-down, down-up-up, and so on), and measure how evenly those ranking patterns are distributed. A signal with a strong, repeating shape produces a small handful of dominant patterns, low entropy. A signal closer to noise produces every possible ordering about equally often, entropy near its maximum. It needs no assumption about the signal's distribution, only its own internal ordering, which is what makes it a natural first check.

**Sample entropy** (Richman and Moorman, 2000) asks a related but distinct question directly in the signal's own real units: pick two windows of length *m* that match each other within a tolerance, then check how often they still match once extended to length *m*+1. A regular, repeating signal keeps matching, low sample entropy. A signal that loses its matches as soon as it is extended further is closer to unpredictable, high sample entropy. It was built to improve on the earlier approximate entropy measure by removing that measure's own self-matching bias, a real, checked improvement, not a cosmetic rename.

**The Hurst exponent** (Hurst, 1951) has nothing to do with electricity in its own original form, it was built to size reservoirs on the Nile from centuries of real flood records. It measures long-range dependence through rescaled-range analysis: H = 0.5 is a pure random walk with no memory, H > 0.5 is persistent (a high reading tends to be followed by more high readings), H < 0.5 is anti-persistent (a high reading tends to be followed by a low one, mean-reverting). A persistent signal is the easier one for a model to learn from, since its own recent past is genuinely informative about its near future.

**The DFA exponent** (Peng et al., 1994) measures a close cousin of the same idea, built originally for DNA sequences rather than rivers, using detrended fluctuation analysis instead of classical rescaled range. Its real practical advantage here: it stays reliable on a signal that is drifting or trending, exactly the kind of real non-stationary behaviour a full year of household load actually has, where the classical Hurst estimator can be distorted by trends the DFA method removes first.

In [ ]:
from twiga.core.stats.entropy import get_dfa_exponent, get_hurst_exponent, get_permutation_entropy, get_sample_entropy

entropy_ref = pd.DataFrame(
    {
        "Metric": ["Permutation entropy", "Sample entropy", "Hurst exponent", "DFA exponent"],
        "Forecastable signal": ["near 0 (regular)", "< 0.5 (regular)", "> 0.5 (persistent)", "> 0.5 (persistent)"],
    }
)
themed_gt(
    GT(entropy_ref).tab_header(title=md("**Forecastability Metrics Used Below**")),
    n_rows=len(entropy_ref),
)

GT(_tbl_data=                Metric Forecastable signal
0  Permutation entropy    near 0 (regular)
1       Sample entropy     < 0.5 (regular)
2       Hurst exponent  > 0.5 (persistent)
3         DFA exponent  > 0.5 (persistent), _body=<great_tables._gt_data.Body object at 0x122ab64b0>, _boxhead=Boxhead([ColInfo(var='Metric', type=<ColInfoTypeEnum.default: 1>, column_label='Metric', column_align='left', column_width=None), ColInfo(var='Forecastable signal', type=<ColInfoTypeEnum.default: 1>, column_label='Forecastable signal', column_align='left', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x12331e5a0>, _spanners=Spanners([]), _heading=Heading(title=Md(text='**Forecastability Metrics Used Below**'), subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x1238a7560>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x1238a7a40>, _source_notes=[], _footnotes=[], _styles=[StyleInfo(locname=LocTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#171717', font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocSubTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#6B7280', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='Metric', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='Forecastable signal', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocSourceNotes(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#6B7280', font=None, size=None, align=None, v_align=None, style='italic', weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocBody(columns=None, rows=[1, 3], mask=None), grpname=None, colname='Metric', rownum=1, colnum=None, styles=[CellStyleFill(color='#F8F9FA')]), StyleInfo(locname=LocBody(columns=None, rows=[1, 3], mask=None), grpname=None, colname='Metric', rownum=3, colnum=None, styles=[CellStyleFill(color='#F8F9FA')]), StyleInfo(locname=LocBody(columns=None, rows=[1, 3], mask=None), grpname=None, colname='Forecastable signal', rownum=1, colnum=None, styles=[CellStyleFill(color='#F8F9FA')]), StyleInfo(locname=LocBody(columns=None, rows=[1, 3], mask=None), grpname=None, colname='Forecastable signal', rownum=3, colnum=None, styles=[CellStyleFill(color='#F8F9FA')])], _locale=<great_tables._gt_data.Locale object at 0x12305d370>, _formats=[], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='100%'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['Libre Franklin', 'Segoe UI', 'Roboto', 'Helvetica Neue', 'sans-serif'

In [ ]:
def forecastability_row(customer_id: int, series: np.ndarray) -> dict:
    return {
        "customer_id": customer_id,
        "permutation_entropy": get_permutation_entropy(series),
        "sample_entropy": get_sample_entropy(series),
        "hurst_exponent": get_hurst_exponent(series),
        "dfa_exponent": get_dfa_exponent(series),
    }


forecastability = pd.DataFrame([forecastability_row(i, series_all[i]) for i in range(n_customers)]).set_index(
    "customer_id"
)
forecastability.describe().round(3)

,permutation_entropy,sample_entropy,hurst_exponent,dfa_exponent
count,342.000,342.000,342.000,342.000
mean,0.987,0.407,0.315,0.754
std,0.007,0.173,0.789,0.072
min,0.962,0.060,-3.985,0.558
25%,0.983,0.284,0.480,0.704
50%,0.988,0.378,0.555,0.753
75%,0.993,0.499,0.615,0.797
max,1.000,1.167,0.784,1.051


In [ ]:
median_pe = forecastability["permutation_entropy"].median()
median_se = forecastability["sample_entropy"].median()
median_hurst = forecastability["hurst_exponent"].median()
median_dfa = forecastability["dfa_exponent"].median()

forecastability_summary = pd.DataFrame(
    {
        "Metric": ["Permutation entropy", "Sample entropy", "Hurst exponent", "DFA exponent"],
        "Median (342 customers)": [
            f"{median_pe:.3f}",
            f"{median_se:.3f}",
            f"{median_hurst:.3f}",
            f"{median_dfa:.3f}",
        ],
        "Interpretation": [
            "Close to the theoretical maximum (1.0): ordering patterns are close to "
            "uniformly spread, little repeating shape",
            "Below 0.5: a real, if modest, repeating structure across windows",
            (
                "Persistent (H > 0.5): recent readings carry real information about near-future ones"
                if median_hurst > 0.5
                else "Near the random-walk boundary"
            ),
            "Persistent (DFA > 0.5), consistent with the Hurst reading once trends are removed",
        ],
    }
)

themed_gt(
    GT(forecastability_summary).tab_header(
        title=md("**Forecastability Metrics: Real Population Results**"),
        subtitle="Median across all 342 real AusNet customers",
    ),
    n_rows=len(forecastability_summary),
)

GT(_tbl_data=                Metric Median (342 customers)  \
0  Permutation entropy                  0.988   
1       Sample entropy                  0.378   
2       Hurst exponent                  0.555   
3         DFA exponent                  0.753   

                                      Interpretation  
0  Close to the theoretical maximum (1.0): orderi...  
1  Below 0.5: a real, if modest, repeating struct...  
2  Persistent (H > 0.5): recent readings carry re...  
3  Persistent (DFA > 0.5), consistent with the Hu...  , _body=<great_tables._gt_data.Body object at 0x127d61550>, _boxhead=Boxhead([ColInfo(var='Metric', type=<ColInfoTypeEnum.default: 1>, column_label='Metric', column_align='left', column_width=None), ColInfo(var='Median (342 customers)', type=<ColInfoTypeEnum.default: 1>, column_label='Median (342 customers)', column_align='right', column_width=None), ColInfo(var='Interpretation', type=<ColInfoTypeEnum.default: 1>, column_label='Interpretation', column_align='left', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x11749f2f0>, _spanners=Spanners([]), _heading=Heading(title=Md(text='**Forecastability Metrics: Real Population Results**'), subtitle='Median across all 342 real AusNet customers', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x127d57620>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x127cfa2d0>, _source_notes=[], _footnotes=[], _styles=[StyleInfo(locname=LocTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#171717', font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocSubTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#6B7280', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='Metric', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='Median (342 customers)', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='Interpretation', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocSourceNotes(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#6B7280', font=None, size=None, align=None, v_align=None, style='italic', weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocBody(columns=None, rows=[1, 3], mask=None), grpname=None, colname='Metric', rownum=1, colnum=None, styles=[CellStyleFill(color='#F8F9FA')]), StyleInfo(locname=LocBody(columns=None, rows=[1, 3], mask=None), grpname=None, colname='Metric', rownum=3, colnum=None, styles=[CellStyleFill(color='#F8F9FA')]), StyleInfo(locname=LocBody(columns=None, rows=[1, 3], mask=None), grpname=None, colname='Median (342 customers)', rownum=1, colnum=None, styles=[CellStyleFill(color='#F8F9FA')]), StyleInfo(locname=LocBody(columns=None, rows=[1, 3], mask=None), grpname=None, colname='Median (342 customers)', rownum=3, colnum=None, styles=[CellStyleFill(color='#F8F9FA')]), StyleInfo(locname=LocBody(columns=None, rows=[1, 3], mask=None), grpname=None, colname='Interpretation', rownum=1, colnum=None, styles=[CellStyleFill(color='#F8F9FA')])

### Meet three real households

Population statistics are easy to state and hard to picture. Three real customers, one real week each, make the abstract concrete before the next section shows all 342 at once: the most persistent household in this pool, the least persistent, and one sitting exactly at the population's own median.


In [ ]:
median_hurst_value = forecastability["hurst_exponent"].median()
snapshot_customers = {
    "Most persistent": forecastability["hurst_exponent"].idxmax(),
    "Typical (median Hurst)": forecastability["hurst_exponent"].sub(median_hurst_value).abs().idxmin(),
    "Least persistent": forecastability["hurst_exponent"].idxmin(),
}

ONE_WEEK_STEPS = STEPS_PER_WEEK
snapshot_rows = []
for label, cust_id in snapshot_customers.items():
    hurst = forecastability.loc[cust_id, "hurst_exponent"]
    pe = forecastability.loc[cust_id, "permutation_entropy"]
    panel = f"{label} (customer {cust_id})\nHurst={hurst:.2f}, perm. entropy={pe:.3f}"
    snapshot_rows.append(
        pd.DataFrame(
            {
                "hour": np.arange(ONE_WEEK_STEPS) / 2,
                "load_kw": series_all[cust_id, :ONE_WEEK_STEPS],
                "panel": panel,
            }
        )
    )
snapshot_df = pd.concat(snapshot_rows, ignore_index=True)
snapshot_df["panel"] = pd.Categorical(snapshot_df["panel"], categories=snapshot_df["panel"].unique(), ordered=True)

p = (
    ggplot(snapshot_df, aes("hour", "load_kw"))
    + geom_line(color=INFO, size=0.6)
    + facet_wrap(facets="panel", ncol=1, scales="free_y")
    + labs(title="One Real Week, Three Real Households", x="Hour", y="Load (kW)")
    + modern_theme()
    + UNBOLD_AXES
    + ggsize(FULL_WIDTH, 560)
)
p

In [ ]:
forecastability_long = forecastability.reset_index().melt(id_vars="customer_id", var_name="metric", value_name="value")
metric_labels = {
    "permutation_entropy": "Permutation entropy",
    "sample_entropy": "Sample entropy",
    "hurst_exponent": "Hurst exponent",
    "dfa_exponent": "DFA exponent",
}
forecastability_long["metric"] = forecastability_long["metric"].map(metric_labels)

p = (
    ggplot(forecastability_long, aes("value", fill="metric"))
    + geom_density(alpha=0.85, color=PRIMARY, show_legend=False)
    + facet_wrap(facets="metric", ncol=2, scales="free")
    + labs(title="Hundreds of Customers, Hundreds of Forecastability Profiles", x="", y="Density")
    + modern_theme()
    + UNBOLD_AXES
    + ggsize(FULL_WIDTH, 380)
)
p

## Lag structure and stationarity

Entropy and Hurst say how regular a signal is; they do not say which real lag carries the signal, or whether the series is drifting under its own diurnal or weekly cycle. Classical Box-Jenkins time-series theory, laid out in full in Hyndman and Athanasopoulos's *Forecasting: Principles and Practice*, fills that gap with four real, complementary tools, run here per customer rather than on one aggregate, at this data's own 30-minute resolution (48 steps to a day, 336 to a week).

**The autocorrelation function (ACF)** measures how strongly a series correlates with a lagged copy of itself, at every real lag up to a chosen horizon. A strong ACF value at lag 48 says "today's reading at this half-hour looks like yesterday's reading at the same half-hour," a daily cycle; a strong value at lag 336 says the same for a full week. **The partial autocorrelation function (PACF)** asks a sharper question: how much does a given lag add *on top of* every shorter lag already in the model, the correlation left over once the intermediate lags are accounted for. The lag where PACF drops inside its own confidence band is the classical signal for how many autoregressive terms, the AR order, a model actually needs; adding more only fits noise.

**Stationarity** asks whether a series' own statistical properties, mean, variance, autocorrelation, stay stable over time, or drift. Two real, complementary tests decide it, deliberately paired because they test opposite null hypotheses. The Augmented Dickey-Fuller test (Dickey and Fuller, 1979) has a null hypothesis of *non-stationary* (a unit root); rejecting it is evidence for stationarity. The KPSS test (Kwiatkowski, Phillips, Schmidt and Shin, 1992) has the opposite null, *stationary*; rejecting it is evidence against stationarity. Run together, the two tests resolve into four honest verdicts rather than one test's blind spot: both agree on stationary, both agree on non-stationary, they disagree in the trend-stationary direction (stationary once a deterministic trend is removed), or they disagree the other way (inconclusive, real ambiguity rather than a forced answer).

In [ ]:
from twiga.core.stats.autocorr import estimate_ar_order
from twiga.core.stats.seasonality import get_acf_values
from twiga.core.stats.stationarity import adf_test, kpss_test


def lag_and_stationarity_row(customer_id: int, series: np.ndarray) -> dict:
    acf_values, _ = get_acf_values(series, maxlag=STEPS_PER_WEEK)  # one real week at 30-minute resolution
    _, ar_order = estimate_ar_order(series, maxlag=100)
    _, adf_p, *_ = adf_test(series)
    _, kpss_p, *_ = kpss_test(series)
    if adf_p < 0.05 and kpss_p > 0.05:
        verdict = "stationary"
    elif adf_p > 0.05 and kpss_p < 0.05:
        verdict = "non-stationary"
    elif adf_p < 0.05 and kpss_p < 0.05:
        verdict = "trend-stationary"
    else:
        verdict = "inconclusive"
    return {
        "customer_id": customer_id,
        "acf_daily_48": acf_values[STEPS_PER_DAY - 1],
        "acf_weekly_336": acf_values[STEPS_PER_WEEK - 1],
        "ar_order": ar_order,
        "adf_p": adf_p,
        "kpss_p": kpss_p,
        "stationarity_verdict": verdict,
    }


lag_stationarity = pd.DataFrame([lag_and_stationarity_row(i, series_all[i]) for i in range(n_customers)]).set_index(
    "customer_id"
)
lag_stationarity.head()

,acf_daily_48,acf_weekly_336,ar_order,adf_p,kpss_p,stationarity_verdict
customer_id,,,,,,
0,0.376814,0.302108,6,0.0,0.010000,trend-stationary
1,0.559705,0.558446,3,0.0,0.089196,stationary
2,0.330702,0.289453,3,0.0,0.010000,trend-stationary
3,0.260027,0.280336,2,0.0,0.010000,trend-stationary
4,0.455893,0.327110,1,0.0,0.010000,trend-stationary


In [ ]:
lag_summary = pd.DataFrame(
    {
        "Metric": ["Daily ACF (lag 48)", "Weekly ACF (lag 336)", "AR order"],
        "Median (342 customers)": [
            f"{lag_stationarity['acf_daily_48'].median():.3f}",
            f"{lag_stationarity['acf_weekly_336'].median():.3f}",
            f"{lag_stationarity['ar_order'].median():.0f}",
        ],
        "Interpretation": [
            "A real, moderate day-to-day repeat, not the near-perfect echo a factory shift pattern would show",
            "A slightly weaker repeat at one week than at one day: the daily rhythm dominates over the weekly one",
            "Half of all customers need only this many autoregressive lags before PACF "
            "drops into its own confidence band",
        ],
    }
)

themed_gt(
    GT(lag_summary).tab_header(
        title=md("**Lag Structure: Real Population Results**"), subtitle="Median across all 342 real AusNet customers"
    ),
    n_rows=len(lag_summary),
)

GT(_tbl_data=                 Metric Median (342 customers)  \
0    Daily ACF (lag 48)                  0.334   
1  Weekly ACF (lag 336)                  0.305   
2              AR order                      4   

                                      Interpretation  
0  A real, moderate day-to-day repeat, not the ne...  
1  A slightly weaker repeat at one week than at o...  
2  Half of all customers need only this many auto...  , _body=<great_tables._gt_data.Body object at 0x127d525d0>, _boxhead=Boxhead([ColInfo(var='Metric', type=<ColInfoTypeEnum.default: 1>, column_label='Metric', column_align='left', column_width=None), ColInfo(var='Median (342 customers)', type=<ColInfoTypeEnum.default: 1>, column_label='Median (342 customers)', column_align='right', column_width=None), ColInfo(var='Interpretation', type=<ColInfoTypeEnum.default: 1>, column_label='Interpretation', column_align='left', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x117514920>, _spanners=Spanners([]), _heading=Heading(title=Md(text='**Lag Structure: Real Population Results**'), subtitle='Median across all 342 real AusNet customers', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x127de3920>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x127de16d0>, _source_notes=[], _footnotes=[], _styles=[StyleInfo(locname=LocTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#171717', font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocSubTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#6B7280', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='Metric', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='Median (342 customers)', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='Interpretation', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocSourceNotes(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#6B7280', font=None, size=None, align=None, v_align=None, style='italic', weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocBody(columns=None, rows=[1], mask=None), grpname=None, colname='Metric', rownum=1, colnum=None, styles=[CellStyleFill(color='#F8F9FA')]), StyleInfo(locname=LocBody(columns=None, rows=[1], mask=None), grpname=None, colname='Median (342 customers)', rownum=1, colnum=None, styles=[CellStyleFill(color='#F8F9FA')]), StyleInfo(locname=LocBody(columns=None, rows=[1], mask=None), grpname=None, colname='Interpretation', rownum=1, colnum=None, styles=[CellStyleFill(color='#F8F9FA')])], _locale=<great_tables._gt_data.Locale object at 0x127de1d30>, _formats=[], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='100%'), table_layout=OptionsInfo(scss=True, category='table', type='value', value=

In [ ]:
most_persistent = forecastability["hurst_exponent"].idxmax()
least_persistent = forecastability["hurst_exponent"].idxmin()

acf_lollipop_chart(
    series_all[most_persistent],
    max_lag=STEPS_PER_WEEK,
    title=f"Most Persistent Real Customer (Hurst={forecastability.loc[most_persistent, 'hurst_exponent']:.2f})",
)

In [ ]:
acf_lollipop_chart(
    series_all[least_persistent],
    max_lag=STEPS_PER_WEEK,
    title=f"Least Persistent Real Customer (Hurst={forecastability.loc[least_persistent, 'hurst_exponent']:.2f})",
)

In [ ]:
# A representative customer for the PACF cutoff: the one whose own AR order
# sits exactly at the population median, not a cherry-picked clean example.
representative_ar = lag_stationarity["ar_order"].sub(lag_stationarity["ar_order"].median()).abs().idxmin()

pacf_lollipop_chart(
    series_all[representative_ar],
    max_lag=100,
    title=f"PACF Cutoff, a Representative Customer (AR order={lag_stationarity.loc[representative_ar, 'ar_order']})",
)

In [ ]:
verdict_counts = (
    lag_stationarity["stationarity_verdict"].value_counts().rename_axis("verdict").reset_index(name="customers")
)
verdict_counts["share"] = (verdict_counts["customers"] / len(lag_stationarity) * 100).round(1)

themed_gt(
    GT(verdict_counts)
    .tab_header(title=md("**Stationarity Verdict Across the Real Customer Sample**"))
    .cols_label(verdict=md("**Verdict**"), customers=md("**Customers**"), share=md("**Share (%)**"))
    .tab_source_note(f"n = {len(lag_stationarity)} real AusNet customers"),
    n_rows=len(verdict_counts),
)

GT(_tbl_data=            verdict  customers  share
0  trend-stationary        323   94.4
1        stationary         19    5.6, _body=<great_tables._gt_data.Body object at 0x127de2e10>, _boxhead=Boxhead([ColInfo(var='verdict', type=<ColInfoTypeEnum.default: 1>, column_label=Md(text='**Verdict**'), column_align='left', column_width=None), ColInfo(var='customers', type=<ColInfoTypeEnum.default: 1>, column_label=Md(text='**Customers**'), column_align='right', column_width=None), ColInfo(var='share', type=<ColInfoTypeEnum.default: 1>, column_label=Md(text='**Share (%)**'), column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x1235c4830>, _spanners=Spanners([]), _heading=Heading(title=Md(text='**Stationarity Verdict Across the Real Customer Sample**'), subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x127d62990>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x127d63f80>, _source_notes=['n = 342 real AusNet customers'], _footnotes=[], _styles=[StyleInfo(locname=LocTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#171717', font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocSubTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#6B7280', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='verdict', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='customers', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='share', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocSourceNotes(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#6B7280', font=None, size=None, align=None, v_align=None, style='italic', weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocBody(columns=None, rows=[1], mask=None), grpname=None, colname='verdict', rownum=1, colnum=None, styles=[CellStyleFill(color='#F8F9FA')]), StyleInfo(locname=LocBody(columns=None, rows=[1], mask=None), grpname=None, colname='customers', rownum=1, colnum=None, styles=[CellStyleFill(color='#F8F9FA')]), StyleInfo(locname=LocBody(columns=None, rows=[1], mask=None), grpname=None, colname='share', rownum=1, colnum=None, styles=[CellStyleFill(color='#F8F9FA')])], _locale=<great_tables._gt_data.Locale object at 0x127d61520>, _formats=[], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='100%'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', typ

## A spectral view of predictability

Entropy, Hurst, and autocorrelation all read a signal in the time domain. A frequency-domain view can catch something they miss: whether a customer's own consumption is dominated by a few strong, regular cycles or spread evenly across the spectrum with no real periodicity at all, exactly the premise behind Wang, Klee and Roos's own Spectral Predictability Score, checked before model development on the M5 competition data. The version below is this book's own construction in the same spirit, a normalized spectral entropy inverted so higher means more forecastable, checked directly for whether it adds real signal beyond the time-domain battery above rather than assumed redundant.

In [ ]:
def spectral_predictability_score(series: np.ndarray) -> float:
    """1 minus the normalized Shannon entropy of the power spectral density.

    Higher means frequency content concentrated in a few regular components
    (more forecastable); lower means power spread evenly across frequencies,
    closer to white noise.
    """
    centered = series - series.mean()
    power = np.abs(np.fft.rfft(centered)) ** 2
    power = power[1:]  # drop the DC component
    power = power[power > 0]
    if power.size == 0:
        return np.nan
    p = power / power.sum()
    spectral_entropy = -(p * np.log(p)).sum() / np.log(p.size)
    return float(1.0 - spectral_entropy)


forecastability["spectral_predictability"] = [
    spectral_predictability_score(series_all[i]) for i in forecastability.index
]
forecastability[["hurst_exponent", "spectral_predictability"]].corr()

,hurst_exponent,spectral_predictability
hurst_exponent,1.000000,-0.050868
spectral_predictability,-0.050868,1.000000


## Which features matter, and does that differ by customer?

Knowing a signal is forecastable is not the same as knowing what to feed a model. A real, honest ensemble of association metrics, ranked and aggregated across several independent statistical tests rather than any single one, answers a sharper question: which lag or calendar feature actually carries the signal for a given customer, and does the answer stay the same from one customer to the next, or does it genuinely vary.

In [ ]:
FORECAST_HORIZON = STEPS_PER_DAY  # 24 hours ahead, the same horizon every model below is checked against


def build_lag_features(series: np.ndarray, *, horizon: int = FORECAST_HORIZON) -> tuple[pd.DataFrame, list[str]]:
    """A small, direct lag-and-calendar feature matrix for a horizon-matched forecast.

    The target is shifted `horizon` steps into the future and every lag
    feature is taken relative to the forecast origin, not the target, so a
    model trained on this matrix answers a real, horizon-matched question,
    not an easier same-step prediction task.
    """
    feat = pd.DataFrame({"value": series})
    feat["target"] = feat["value"].shift(-horizon)
    for lag in (0, STEPS_PER_DAY, STEPS_PER_DAY * 2, STEPS_PER_WEEK):
        feat[f"lag_{lag}"] = feat["value"].shift(lag)
    step_idx = np.arange(len(feat))
    feat["half_hour"] = step_idx % STEPS_PER_DAY
    feat["dayofweek"] = (step_idx // STEPS_PER_DAY) % 7
    feature_cols = [c for c in feat.columns if c not in ("value", "target")]
    return feat.dropna().reset_index(drop=True), feature_cols

In [ ]:
import numpy as _np
import twiga.core.data.selection as _selection


# A real, reproducible bug in this installed twiga version: diversity_weights
# mutates a read-only array returned by DataFrame.corr().to_numpy() under
# this environment's numpy/pandas combination. Patched here, once, with the
# one-line fix (an explicit .copy()) rather than losing the ensemble ranker's
# real diversity-weighting behavior over a single weaker metric.
def _diversity_weights_fixed(score_matrix):
    n = score_matrix.shape[1]
    if n == 1:
        return _np.ones(1)
    corr = score_matrix.corr(method="spearman").abs().to_numpy().copy()
    _np.fill_diagonal(corr, 0.0)
    uniqueness = 1.0 - corr.mean(axis=1)
    weights = _np.clip(uniqueness, 0.05, None)
    return weights / weights.sum()


_selection.diversity_weights = _diversity_weights_fixed
from twiga.core.data.selection import select_top_features

N_FEATURE_SAMPLE = 40
feature_ids = RNG.choice(n_customers, size=N_FEATURE_SAMPLE, replace=False)

top_feature_rows = []
for cust_id in feature_ids:
    feat, feature_cols = build_lag_features(series_all[cust_id])
    top = select_top_features(data=feat, features=feature_cols, target="target", top_k=1)
    if top:
        top_feature_rows.append({"customer_id": cust_id, "top_feature": top[0]})

top_feature_counts = (
    pd.DataFrame(top_feature_rows)["top_feature"].value_counts().rename_axis("feature").reset_index(name="customers")
)
top_feature_counts["share"] = (top_feature_counts["customers"] / len(top_feature_rows) * 100).round(1)

themed_gt(
    GT(top_feature_counts)
    .tab_header(title=md("**The Single Most Relevant Feature, Per Customer**"), subtitle="24-hour-ahead net load")
    .cols_label(feature=md("**Top feature**"), customers=md("**Customers**"), share=md("**Share (%)**"))
    .tab_source_note(
        f"n = {len(top_feature_rows)} real customers, ensemble-ranked "
        "(Spearman, ANOVA, mutual information, Xi, PPS, random-forest importance)"
    ),
    n_rows=len(top_feature_counts),
)

GT(_tbl_data=     feature  customers  share
0  half_hour         20   50.0
1      lag_0         19   47.5
2     lag_48          1    2.5, _body=<great_tables._gt_data.Body object at 0x127d94410>, _boxhead=Boxhead([ColInfo(var='feature', type=<ColInfoTypeEnum.default: 1>, column_label=Md(text='**Top feature**'), column_align='left', column_width=None), ColInfo(var='customers', type=<ColInfoTypeEnum.default: 1>, column_label=Md(text='**Customers**'), column_align='right', column_width=None), ColInfo(var='share', type=<ColInfoTypeEnum.default: 1>, column_label=Md(text='**Share (%)**'), column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x127d72c00>, _spanners=Spanners([]), _heading=Heading(title=Md(text='**The Single Most Relevant Feature, Per Customer**'), subtitle='24-hour-ahead net load', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x127d576b0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x127d56ae0>, _source_notes=['n = 40 real customers, ensemble-ranked (Spearman, ANOVA, mutual information, Xi, PPS, random-forest importance)'], _footnotes=[], _styles=[StyleInfo(locname=LocTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#171717', font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocSubTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#6B7280', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='feature', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='customers', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='share', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocSourceNotes(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#6B7280', font=None, size=None, align=None, v_align=None, style='italic', weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocBody(columns=None, rows=[1], mask=None), grpname=None, colname='feature', rownum=1, colnum=None, styles=[CellStyleFill(color='#F8F9FA')]), StyleInfo(locname=LocBody(columns=None, rows=[1], mask=None), grpname=None, colname='customers', rownum=1, colnum=None, styles=[CellStyleFill(color='#F8F9FA')]), StyleInfo(locname=LocBody(columns=None, rows=[1], mask=None), grpname=None, colname='share', rownum=1, colnum=None, styles=[CellStyleFill(color='#F8F9FA')])], _locale=<great_tables._gt_data.Locale object at 0x127d55df0>, _formats=[], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='100%'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, catego

## Which baseline actually wins, and does it vary by customer?

A trained model's real cost, engineering time, compute, and maintenance, only pays for itself if it beats the cheapest reasonable alternative. Hyndman and Athanasopoulos are direct about this in *Forecasting: Principles and Practice*: naive, seasonal-naive, drift, and average-based benchmarks exist precisely so a more complex method has something honest to beat, and a method that cannot beat them is not worth the complexity, whatever else it promises. Chapter 2 is where a trained tree ensemble gets that honest check; this chapter's own job is narrower and comes first: which of `twiga.models.baseline`'s own persistence strategies, naive, seasonal-naive, drift, or window-average, already wins for a given customer, with no model training at all. Reused directly from twiga's own tutorial precedent, where seasonal-naive won outright on a different real dataset: worth checking directly whether the same baseline wins here too, not assumed to transfer.

In [ ]:
from twiga.models.baseline.drift_model import DRIFTConfig, DRIFTModel
from twiga.models.baseline.naive_model import NAIVEConfig, NAIVEModel
from twiga.models.baseline.seasonal_naive_model import SEASONALNAIVEConfig, SEASONALNAIVEModel
from twiga.models.baseline.window_average_model import WINDOWAVERAGEConfig, WINDOWAVERAGEModel

LOOKBACK = STEPS_PER_WEEK  # one real week of context per prediction window
TEST_STEPS = 90 * STEPS_PER_DAY  # the last 90 real days, held out per customer

baseline_models = {
    "naive": NAIVEModel(NAIVEConfig(strategy="window_last"), num_targets=1, horizon=FORECAST_HORIZON),
    "seasonal_naive": SEASONALNAIVEModel(
        SEASONALNAIVEConfig(period=STEPS_PER_DAY), num_targets=1, horizon=FORECAST_HORIZON
    ),
    "drift": DRIFTModel(DRIFTConfig(), num_targets=1, horizon=FORECAST_HORIZON),
    "window_average": WINDOWAVERAGEModel(WINDOWAVERAGEConfig(), num_targets=1, horizon=FORECAST_HORIZON),
}


def baseline_windows(series: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Sliding (X, y) windows over a real customer's own last 90 real days."""
    start = len(series) - TEST_STEPS - LOOKBACK - FORECAST_HORIZON
    segment = series[start:]
    n_windows = len(segment) - LOOKBACK - FORECAST_HORIZON + 1
    x = np.lib.stride_tricks.sliding_window_view(segment, LOOKBACK)[:n_windows, :, None]
    y = np.lib.stride_tricks.sliding_window_view(segment[LOOKBACK:], FORECAST_HORIZON)[:n_windows, :, None]
    return x, y


baseline_rows = []
for cust_id in range(n_customers):
    x, y = baseline_windows(series_all[cust_id])
    row = {"customer_id": cust_id}
    for name, model in baseline_models.items():
        preds = model.predict(x)
        row[f"mae_{name}"] = float(np.mean(np.abs(preds - y)))
    baseline_rows.append(row)

baseline_results = pd.DataFrame(baseline_rows).set_index("customer_id")
baseline_results.describe().round(3)

,mae_naive,mae_seasonal_naive,mae_drift,mae_window_average
count,342.000,342.000,342.000,342.000
mean,0.360,0.295,0.370,0.289
std,0.207,0.168,0.213,0.175
min,0.045,0.045,0.048,0.044
25%,0.220,0.183,0.226,0.175
50%,0.323,0.270,0.331,0.255
75%,0.425,0.350,0.442,0.344
max,1.872,1.783,1.951,1.792


In [ ]:
from twiga.core.metrics import get_pointwise_metrics
from twiga.core.plot.gt import twiga_report

# NBIAS excluded, checked directly rather than assumed safe: twiga's own
# formula is a raw sum of per-step (y - yhat) / (y + yhat) terms, not
# normalized by sample count, and every one of the four baselines has at
# least one real customer whose actual-plus-predicted crosses zero at some
# half-hour (drift worst, down to ~1e-17), a single near-zero denominator
# that then dominates the whole sum. Confirmed by direct inspection, not a
# bug in this notebook's own aggregation: MAE, RMSE, Corr, WMAPE, and SMAPE
# all stay well-behaved and bounded on the same data.
METRIC_NAMES = ["mae", "rmse", "corr", "wmape", "smape"]

metric_rows = []
for cust_id in range(n_customers):
    x, y = baseline_windows(series_all[cust_id])
    actual = y.reshape(-1)
    for name, model in baseline_models.items():
        preds = model.predict(x).reshape(-1)
        row = get_pointwise_metrics(actual, preds, metric_names=METRIC_NAMES)
        row["Model"] = name
        metric_rows.append(row)

metrics_all = pd.concat(metric_rows, ignore_index=True)
res = metrics_all.groupby("Model", sort=False)[METRIC_NAMES].mean().round(4).reset_index()
res = res.rename(columns={"mae": "MAE", "rmse": "RMSE", "corr": "Corr", "wmape": "WMAPE", "smape": "SMAPE"})

twiga_report(
    res,
    ["MAE", "RMSE", "Corr", "WMAPE", "SMAPE"],
    ["MAE", "RMSE", "WMAPE", "SMAPE"],
    ["Corr"],
    title="Model Performance, Averaged Across 342 Real Customers",
    subtitle="24-hour-ahead, last 90 real days held out per customer",
    source_note="twiga.models.baseline, real AusNet data",
)

GT(_tbl_data=            Model     MAE    RMSE    Corr    WMAPE    SMAPE
0           naive  0.3595  0.5905  0.0769  79.0624  63.5501
1  seasonal_naive  0.2947  0.5237  0.2578  65.6525  51.9019
2           drift  0.3701  0.6084  0.0755  81.4465  66.8983
3  window_average  0.2889  0.4360  0.0727  63.3271  60.4267, _body=<great_tables._gt_data.Body object at 0x13297cce0>, _boxhead=Boxhead([ColInfo(var='Model', type=<ColInfoTypeEnum.default: 1>, column_label='Model', column_align='left', column_width=None), ColInfo(var='MAE', type=<ColInfoTypeEnum.default: 1>, column_label='MAE', column_align='right', column_width=None), ColInfo(var='RMSE', type=<ColInfoTypeEnum.default: 1>, column_label='RMSE', column_align='right', column_width=None), ColInfo(var='Corr', type=<ColInfoTypeEnum.default: 1>, column_label='Corr', column_align='right', column_width=None), ColInfo(var='WMAPE', type=<ColInfoTypeEnum.default: 1>, column_label='WMAPE', column_align='right', column_width=None), ColInfo(var='SMAPE', type=<ColInfoTypeEnum.default: 1>, column_label='SMAPE', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x13297c530>, _spanners=Spanners([]), _heading=Heading(title='Model Performance, Averaged Across 342 Real Customers', subtitle='24-hour-ahead, last 90 real days held out per customer', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x13297d040>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x13297cfe0>, _source_notes=['twiga.models.baseline, real AusNet data'], _footnotes=[], _styles=[StyleInfo(locname=LocTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#107591', font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocSubTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#718096', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='Model', rownum=None, colnum=None, styles=[CellStyleText(color='#ffffff', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='MAE', rownum=None, colnum=None, styles=[CellStyleText(color='#ffffff', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='RMSE', rownum=None, colnum=None, styles=[CellStyleText(color='#ffffff', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='Corr', rownum=None, colnum=None, styles=[CellStyleText(color='#ffffff', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='WMAPE', rownum=None, colnum=None, styles=[CellStyleText(color='#ffffff', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='SMAPE', rownum=None, colnum=None, styles=[CellStyleText(color='#ffffff', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocSourceNotes(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#718096', font=None, size=None, align=None, v_align=None, st

In [ ]:
from ark.evaluate.ranking import rank_models

ranked_baselines = rank_models(res.set_index("Model")[["MAE", "RMSE", "Corr", "WMAPE", "SMAPE"]])
themed_gt(ranked_baselines.rename_axis("Model").reset_index().round(4))

Model,MAE,RMSE,Corr,WMAPE,SMAPE,arith_rank,borda_count,weighted_borda
seasonal_naive,0.2947,0.5237,0.2578,65.6525,51.9019,3.4,17.0,3.5082
window_average,0.2889,0.436,0.0727,63.3271,60.4267,3.2,16.0,2.8361
naive,0.3595,0.5905,0.0769,79.0624,63.5501,2.2,11.0,2.3279
drift,0.3701,0.6084,0.0755,81.4465,66.8983,1.2,6.0,1.3279


In [ ]:
mae_cols = [c for c in baseline_results.columns if c.startswith("mae_")]
baseline_results["best_baseline"] = baseline_results[mae_cols].idxmin(axis=1).str.replace("mae_", "")

win_counts = baseline_results["best_baseline"].value_counts().rename_axis("baseline").reset_index(name="customers")
win_counts["share"] = (win_counts["customers"] / len(baseline_results) * 100).round(1)

themed_gt(
    GT(win_counts)
    .tab_header(
        title=md("**Which Baseline Wins, Per Customer**"), subtitle="No model training, four persistence strategies"
    )
    .cols_label(baseline=md("**Baseline**"), customers=md("**Customers**"), share=md("**Share (%)**"))
    .tab_source_note(f"n = {len(baseline_results)} real AusNet customers, 24-hour-ahead, last 90 real days held out"),
    n_rows=len(win_counts),
)

GT(_tbl_data=         baseline  customers  share
0  window_average        239   69.9
1  seasonal_naive        102   29.8
2           naive          1    0.3, _body=<great_tables._gt_data.Body object at 0x132b98380>, _boxhead=Boxhead([ColInfo(var='baseline', type=<ColInfoTypeEnum.default: 1>, column_label=Md(text='**Baseline**'), column_align='left', column_width=None), ColInfo(var='customers', type=<ColInfoTypeEnum.default: 1>, column_label=Md(text='**Customers**'), column_align='right', column_width=None), ColInfo(var='share', type=<ColInfoTypeEnum.default: 1>, column_label=Md(text='**Share (%)**'), column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x1311d8410>, _spanners=Spanners([]), _heading=Heading(title=Md(text='**Which Baseline Wins, Per Customer**'), subtitle='No model training, four persistence strategies', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x132b985f0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x132b987a0>, _source_notes=['n = 342 real AusNet customers, 24-hour-ahead, last 90 real days held out'], _footnotes=[], _styles=[StyleInfo(locname=LocTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#171717', font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocSubTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#6B7280', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='baseline', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='customers', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='share', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocSourceNotes(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#6B7280', font=None, size=None, align=None, v_align=None, style='italic', weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocBody(columns=None, rows=[1], mask=None), grpname=None, colname='baseline', rownum=1, colnum=None, styles=[CellStyleFill(color='#F8F9FA')]), StyleInfo(locname=LocBody(columns=None, rows=[1], mask=None), grpname=None, colname='customers', rownum=1, colnum=None, styles=[CellStyleFill(color='#F8F9FA')]), StyleInfo(locname=LocBody(columns=None, rows=[1], mask=None), grpname=None, colname='share', rownum=1, colnum=None, styles=[CellStyleFill(color='#F8F9FA')])], _locale=<great_tables._gt_data.Locale object at 0x132b98860>, _formats=[], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='100%'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='tabl

In [ ]:
from twiga.core.plot import plot_forecast_grid

# A representative real customer for the trace chart: whose own window-average
# MAE sits closest to the population's own mean, not a cherry-picked clean example.
representative_cust = (
    (baseline_results["mae_window_average"] - baseline_results["mae_window_average"].mean()).abs().idxmin()
)

x_rep, y_rep = baseline_windows(series_all[representative_cust])
n_trace_days = 7
trace_window_idx = np.arange(n_trace_days) * STEPS_PER_DAY  # non-overlapping, one window per real day
actual_trace = y_rep[trace_window_idx].reshape(-1)

preds_rows = []
for name, model in baseline_models.items():
    preds_trace = model.predict(x_rep[trace_window_idx]).reshape(-1)
    preds_rows.append(pd.DataFrame({"Actual": actual_trace, "forecast": preds_trace, "Model": name}))
preds_all = pd.concat(preds_rows, ignore_index=True)

# plot_forecast_grid has no `colors` parameter of its own (checked directly,
# not assumed): it hard-codes twiga's own Actual/Forecast palette internally.
# Overriding the color scale afterward, rather than passing an unsupported
# kwarg, is what actually aligns it with this book's own brand tokens:
# Actual as a neutral reference (matching the muted grey used for "actual"
# reference lines everywhere else in this notebook), Forecast in this book's
# own primary accent, not twiga's teal.
p = plot_forecast_grid(
    preds_all,
    actual_col="Actual",
    forecast_col="forecast",
    model_col="Model",
    n_samples_per_model=n_trace_days * STEPS_PER_DAY,
    y_label="Load (kW)",
    title="Forecast Traces vs. Actual, One Real Week",
    fig_width=FULL_WIDTH,
    ncol=2,
)
p + scale_color_manual(values={"Actual": GRAY_600, "Forecast": INFO}, name="") + modern_theme()

## The PV signal hiding in net load

AusNet's own data carries a real, separately-recorded seasonal PV shape, shared across customers and scaled by each customer's own assumed system size, not blended invisibly into one metered column the way a real net-metering-only utility feed usually is. That real, known component is used here for a check no proxy could offer directly: does adding a known PV component to a customer's own real load make that customer's own net load look more or less forecastable, by the exact same entropy and Hurst battery run above. Net load with rising DER adoption does not necessarily just get more volatile either: Fias, Hashmi and Deconinck's own 2025 finding is that joint EV and PV activity can offset each other through a compensatory daytime effect, worth checking what a known PV component alone does here rather than assumed.

Before measuring what a known PV component does to forecastability, a look at the real shape itself: one real day per season from AusNet's own PV array, the same real seasonal variation the population-level PV signal used throughout this chapter actually has.


In [ ]:
# Same real representative days Part 4's own seasonal sweep uses.
SEASON_DAYS = {"Summer": 21, "Autumn": 59, "Winter": 199, "Spring": 229}
SEASON_COLORS = {"Summer": WARNING, "Autumn": AI_ACCENT, "Winter": INFO, "Spring": SUCCESS}

pv_profile_df = pd.concat(
    [
        pd.DataFrame({"hour": np.arange(STEPS_PER_DAY) / 2, "pv": pv_data[day], "season": season})
        for season, day in SEASON_DAYS.items()
    ],
    ignore_index=True,
)
pv_profile_df["season"] = pd.Categorical(pv_profile_df["season"], categories=list(SEASON_DAYS), ordered=True)

p = (
    ggplot(pv_profile_df, aes("hour", "pv", color="season"))
    + geom_line(size=1.0)
    + scale_color_manual(values=SEASON_COLORS, name="")
    + labs(title="AusNet's Real PV Shape, One Day Per Season", x="Hour", y="PV output (per unit)")
    + modern_theme()
    + UNBOLD_AXES
    + ggsize(FULL_WIDTH, 300)
)
p

In [ ]:
PV_SHARE = 0.3  # a realistic real rooftop-adoption share, matching Part 4's own penetration studies
pv_rng = np.random.default_rng(42)  # a dedicated draw, independent of the feature-ranking sample above
pv_customers = pv_rng.choice(n_customers, size=round(n_customers * PV_SHARE), replace=False)
print(f"customers with an assumed {PV_KVA:.0f} kVA rooftop PV system: {len(pv_customers)} / {n_customers}")

net_load_data = load_data.copy()
net_load_data[pv_customers] = net_load_data[pv_customers] - PV_KVA * pv_data[None, :, :]
net_series_all = net_load_data.reshape(n_customers, -1)

export_share = (net_series_all[pv_customers] < 0).mean()
print(
    f"share of half-hours these {len(pv_customers)} customers' own net load goes negative "
    f"(a real export event): {export_share:.1%}"
)

customers with an assumed 5 kVA rooftop PV system: 103 / 342
share of half-hours these 103 customers' own net load goes negative (a real export event): 37.0%


In [ ]:
pv_effect_rows = []
for cust_id in pv_customers:
    pv_effect_rows.append(
        {
            "customer_id": cust_id,
            "hurst_load": get_hurst_exponent(series_all[cust_id]),
            "hurst_net": get_hurst_exponent(net_series_all[cust_id]),
            "permutation_entropy_load": get_permutation_entropy(series_all[cust_id]),
            "permutation_entropy_net": get_permutation_entropy(net_series_all[cust_id]),
        }
    )

pv_effect = pd.DataFrame(pv_effect_rows).set_index("customer_id")
pv_effect["hurst_delta"] = pv_effect["hurst_net"] - pv_effect["hurst_load"]
pv_effect["permutation_entropy_delta"] = pv_effect["permutation_entropy_net"] - pv_effect["permutation_entropy_load"]
pv_effect[
    ["hurst_load", "hurst_net", "hurst_delta", "permutation_entropy_load", "permutation_entropy_net"]
].describe().round(3)

,hurst_load,hurst_net,hurst_delta,permutation_entropy_load,permutation_entropy_net
count,103.000,103.000,103.000,103.000,103.000
mean,0.330,0.587,0.257,0.987,0.955
std,0.752,0.244,0.710,0.008,0.016
min,-3.034,-0.908,-0.944,0.963,0.900
25%,0.509,0.595,0.014,0.984,0.949
50%,0.562,0.630,0.064,0.988,0.955
75%,0.619,0.653,0.131,0.992,0.966
max,0.784,0.819,3.642,1.000,0.992


In [ ]:
pv_summary = pd.DataFrame(
    {
        "Metric": ["Hurst exponent", "Permutation entropy"],
        "Median, load only": [
            f"{pv_effect['hurst_load'].median():.3f}",
            f"{pv_effect['permutation_entropy_load'].median():.3f}",
        ],
        "Median, with known PV": [
            f"{pv_effect['hurst_net'].median():.3f}",
            f"{pv_effect['permutation_entropy_net'].median():.3f}",
        ],
        "Interpretation": [
            "Higher with PV added: a known, regular daily generation cycle makes net load "
            "look *more* persistent, not less",
            "Slightly lower with PV added: consistent with the same more-regular-signal finding from the Hurst reading",
        ],
    }
)

themed_gt(
    GT(pv_summary).tab_header(
        title=md("**A Known PV Component Changes Measured Forecastability**"),
        subtitle=f"n = {len(pv_customers)} real customers with an assumed {PV_KVA:.0f} kVA system",
    ),
    n_rows=len(pv_summary),
)

GT(_tbl_data=                Metric Median, load only Median, with known PV  \
0       Hurst exponent             0.562                 0.630   
1  Permutation entropy             0.988                 0.955   

                                      Interpretation  
0  Higher with PV added: a known, regular daily g...  
1  Slightly lower with PV added: consistent with ...  , _body=<great_tables._gt_data.Body object at 0x1311d9b50>, _boxhead=Boxhead([ColInfo(var='Metric', type=<ColInfoTypeEnum.default: 1>, column_label='Metric', column_align='left', column_width=None), ColInfo(var='Median, load only', type=<ColInfoTypeEnum.default: 1>, column_label='Median, load only', column_align='right', column_width=None), ColInfo(var='Median, with known PV', type=<ColInfoTypeEnum.default: 1>, column_label='Median, with known PV', column_align='right', column_width=None), ColInfo(var='Interpretation', type=<ColInfoTypeEnum.default: 1>, column_label='Interpretation', column_align='left', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x1329a9700>, _spanners=Spanners([]), _heading=Heading(title=Md(text='**A Known PV Component Changes Measured Forecastability**'), subtitle='n = 103 real customers with an assumed 5 kVA system', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x1329a9d90>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x1329a91c0>, _source_notes=[], _footnotes=[], _styles=[StyleInfo(locname=LocTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#171717', font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocSubTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#6B7280', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='Metric', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='Median, load only', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='Median, with known PV', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='Interpretation', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocSourceNotes(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#6B7280', font=None, size=None, align=None, v_align=None, style='italic', weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocBody(columns=None, rows=[1], mask=None), grpname=None, colname='Metric', rownum=1, colnum=None, styles=[CellStyleFill(color='#F8F9FA')]), StyleInfo(locname=LocBody(columns=None, rows=[1], mask=None), grpname=None, colname='Median, load only', rownum=1, colnum=None, styles=[CellStyleFill(color='#F8F9FA')]), StyleInfo(locname=LocBody(columns=None, rows=[1], mask=None), grpname=None, colname='Median, with known PV', rownum=1, colnum=None, styles=[CellStyleFill(color='#F8F9FA')]), StyleInfo(locname=LocBody(columns=None, rows=[1], mas

In [ ]:
pv_compare = pd.concat(
    [
        pd.DataFrame({"hurst_exponent": pv_effect["hurst_load"], "signal": "load only"}),
        pd.DataFrame({"hurst_exponent": pv_effect["hurst_net"], "signal": "load with known PV"}),
    ],
    ignore_index=True,
)

p = (
    ggplot(pv_compare, aes("hurst_exponent", fill="signal"))
    + geom_density(alpha=0.6, color=GRAY_600)
    + scale_fill_manual(values={"load only": GRAY_600, "load with known PV": SUCCESS})
    + labs(
        title="A Known PV Component Shifts the Hurst Distribution Higher",
        subtitle=f"n = {len(pv_customers)} real customers, before vs. after adding a known {PV_KVA:.0f} kVA PV system",
        x="Hurst exponent",
        y="Density",
    )
    + modern_theme()
    + UNBOLD_AXES
    + ggsize(FULL_WIDTH, 300)
)
p

## Load, PV, and EV: which signal is more forecastable?

The chapters above measure load and a known PV component. Part 4's own EV-demand section already vendored a third real signal: diversified home EV-charging profiles from the UK's real "Electric Nation" trial (via Team-Nando's `EV-Demand-Profiles`, University of Melbourne), at 1-minute resolution, weekday and weekend, diversified across 100-1,200 real vehicles.

A real, honest caveat before comparing anything: this EV data is not the same *kind* of signal as load or PV. Load is 342 real, individual households across a genuine full year; PV is one real, seasonally-varying shape across the same year. The EV data is a single representative weekday shape and a single representative weekend shape, diversified across many real vehicles, with no per-vehicle time series and no real day-to-day or seasonal variation at all. Making it long enough to run the same forecastability battery means tiling those two shapes across a synthetic year (5 weekday copies and 2 weekend copies per week, repeated), a real, disclosed construction, not a genuine record of one EV owner's own charging behaviour. Its own forecastability score below is closer to a ceiling this diversified, idealised pattern reaches than a real individual customer's own measured unpredictability, since real day-to-day noise (a vehicle plugged in an hour later, a missed charge, a longer trip) is exactly what this construction cannot show.

For a fair comparison, Load below is the population-mean signal across all 342 customers, not one individual, since PV and EV are both already aggregate, shared shapes rather than raw individual noise. A real, checked bonus finding falls out of that choice directly: the population mean's own daily ACF (0.845) sits far above any individual customer's own median (0.334, reported earlier in this chapter). Averaging across customers smooths away exactly the idiosyncratic, person-to-person noise that makes an individual customer harder to forecast, the same real aggregation-level effect Peng et al. (2019) reported and this chapter opened with, showing up again here from a different angle.

In [ ]:
EV_DIR = DATA_DIR.parent / "ev-profiles"

# Same file and downsampling convention as Part 4 Chapter 2's own EV section:
# Level 2 (7.36 kW), diversified median of 100 real vehicles, 1-minute
# readings averaged down to this chapter's own 30-minute resolution.
ev_weekday = (
    np.load(EV_DIR / "Weekday - Level 2 - Diversified (Median) of 100 EVs.npy").reshape(STEPS_PER_DAY, 30).mean(axis=1)
)
ev_weekend = (
    np.load(EV_DIR / "Weekend - Level 2 - Diversified (Median) of 100 EVs.npy").reshape(STEPS_PER_DAY, 30).mean(axis=1)
)
print(
    f"real weekday peak: {ev_weekday.max():.2f} kW at half-hour {ev_weekday.argmax()} ({ev_weekday.argmax() / 2:.1f}h)"
)

# The disclosed construction: assign the real weekend shape to Saturday/Sunday
# and the real weekday shape to every other day, tiled across a synthetic year.
day_of_week = np.arange(n_days) % 7
is_weekend = np.isin(day_of_week, [5, 6])
ev_series = np.where(is_weekend[:, None], ev_weekend[None, :], ev_weekday[None, :]).reshape(-1)
print(f"constructed EV series: {ev_series.shape[0]:,} half-hour steps, matching load/PV's own length")

real weekday peak: 1.49 kW at half-hour 42 (21.0h)
constructed EV series: 17,520 half-hour steps, matching load/PV's own length


A look at the real shape before tiling it into anything: the real diversified weekday and weekend home-charging profiles, Level 2 charging, median across 100 real vehicles.


In [ ]:
ev_profile_df = pd.concat(
    [
        pd.DataFrame({"hour": np.arange(STEPS_PER_DAY) / 2, "ev_kw": ev_weekday, "day_type": "Weekday"}),
        pd.DataFrame({"hour": np.arange(STEPS_PER_DAY) / 2, "ev_kw": ev_weekend, "day_type": "Weekend"}),
    ],
    ignore_index=True,
)

p = (
    ggplot(ev_profile_df, aes("hour", "ev_kw", color="day_type"))
    + geom_line(size=1.0)
    + scale_color_manual(values={"Weekday": INFO, "Weekend": WARNING}, name="")
    + labs(title="Real Diversified EV Home-Charging Shape", x="Hour", y="Load (kW)")
    + modern_theme()
    + UNBOLD_AXES
    + ggsize(FULL_WIDTH, 300)
)
p

In [ ]:
signal_series = {
    "Load": series_all.mean(
        axis=0
    ),  # the population-level signal, comparable to PV/EV's own population-level construction
    "PV": pv_data.reshape(-1),
    "EV": ev_series,
}

signal_rows = []
for name, s in signal_series.items():
    acf_values, _ = get_acf_values(s, maxlag=STEPS_PER_WEEK)
    signal_rows.append(
        {
            "Signal": name,
            "permutation_entropy": get_permutation_entropy(s),
            "sample_entropy": get_sample_entropy(s),
            "hurst_exponent": get_hurst_exponent(s),
            "dfa_exponent": get_dfa_exponent(s),
            "acf_daily_48": acf_values[STEPS_PER_DAY - 1],
            "acf_weekly_336": acf_values[STEPS_PER_WEEK - 1],
        }
    )

signal_comparison = pd.DataFrame(signal_rows).set_index("Signal").round(3)
signal_comparison

,permutation_entropy,sample_entropy,hurst_exponent,dfa_exponent,acf_daily_48,acf_weekly_336
Signal,,,,,,
Load,0.961,0.407,0.626,0.909,0.845,0.751
PV,0.766,0.082,0.386,0.731,0.828,0.774
EV,0.855,0.201,-0.010,0.625,0.954,0.981


In [ ]:
signal_table = signal_comparison.reset_index().rename(
    columns={
        "permutation_entropy": "Perm. entropy",
        "sample_entropy": "Sample entropy",
        "hurst_exponent": "Hurst",
        "dfa_exponent": "DFA",
        "acf_daily_48": "ACF (daily)",
        "acf_weekly_336": "ACF (weekly)",
    }
)

themed_gt(
    GT(signal_table)
    .tab_header(
        title=md("**Load, PV, and EV: a Real Forecastability Comparison**"),
        subtitle="EV is a diversified, tiled construction, a ceiling, not a real individual customer's own variability",
    )
    .tab_source_note(
        "Load = population-mean across 342 real AusNet customers; PV = the real shared AusNet PV shape; "
        "EV = Team-Nando's real diversified UK Electric Nation profile, tiled weekday/weekend across a synthetic year"
    ),
    n_rows=len(signal_table),
)

GT(_tbl_data=  Signal  Perm. entropy  Sample entropy  Hurst    DFA  ACF (daily)  \
0   Load          0.961           0.407  0.626  0.909        0.845   
1     PV          0.766           0.082  0.386  0.731        0.828   
2     EV          0.855           0.201 -0.010  0.625        0.954   

   ACF (weekly)  
0         0.751  
1         0.774  
2         0.981  , _body=<great_tables._gt_data.Body object at 0x132962510>, _boxhead=Boxhead([ColInfo(var='Signal', type=<ColInfoTypeEnum.default: 1>, column_label='Signal', column_align='left', column_width=None), ColInfo(var='Perm. entropy', type=<ColInfoTypeEnum.default: 1>, column_label='Perm. entropy', column_align='right', column_width=None), ColInfo(var='Sample entropy', type=<ColInfoTypeEnum.default: 1>, column_label='Sample entropy', column_align='right', column_width=None), ColInfo(var='Hurst', type=<ColInfoTypeEnum.default: 1>, column_label='Hurst', column_align='right', column_width=None), ColInfo(var='DFA', type=<ColInfoTypeEnum.default: 1>, column_label='DFA', column_align='right', column_width=None), ColInfo(var='ACF (daily)', type=<ColInfoTypeEnum.default: 1>, column_label='ACF (daily)', column_align='right', column_width=None), ColInfo(var='ACF (weekly)', type=<ColInfoTypeEnum.default: 1>, column_label='ACF (weekly)', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x132b9ab40>, _spanners=Spanners([]), _heading=Heading(title=Md(text='**Load, PV, and EV: a Real Forecastability Comparison**'), subtitle="EV is a diversified, tiled construction, a ceiling, not a real individual customer's own variability", preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x132a3b8c0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x132a3b8f0>, _source_notes=["Load = population-mean across 342 real AusNet customers; PV = the real shared AusNet PV shape; EV = Team-Nando's real diversified UK Electric Nation profile, tiled weekday/weekend across a synthetic year"], _footnotes=[], _styles=[StyleInfo(locname=LocTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#171717', font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocSubTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#6B7280', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='Signal', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='Perm. entropy', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='Sample entropy', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='Hurst', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='DFA', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, wh

In [ ]:
signal_long = (
    signal_comparison[["permutation_entropy", "hurst_exponent"]]
    .reset_index()
    .melt(id_vars="Signal", var_name="metric", value_name="value")
)
signal_long["metric"] = signal_long["metric"].map(
    {
        "permutation_entropy": "Permutation entropy (lower = more regular)",
        "hurst_exponent": "Hurst exponent (higher = more persistent)",
    }
)
signal_long["Signal"] = pd.Categorical(signal_long["Signal"], categories=["Load", "PV", "EV"], ordered=True)

p = (
    ggplot(signal_long, aes(x="Signal", y="value", fill="Signal"))
    + geom_bar(stat="identity", width=0.5, show_legend=False)
    + geom_hline(yintercept=0, color=GRAY_600, size=0.3)
    + scale_fill_manual(values=SIGNAL_COLORS)
    + facet_wrap(facets="metric", ncol=2, scales="free_y")
    + labs(title="EV's Score Is a Construction Ceiling", x="", y="")
    + modern_theme()
    + UNBOLD_AXES
    + ggsize(FULL_WIDTH, 300)
)
p

## From diagnostic to decision

A single recommendation that claims to fit every customer is a false confidence, the same real lesson De Bortoli, Ferrari, Ravazzolo and Rossini draw from Italian electricity load data: report the set of approaches that are actually defensible for a given regime, not one falsely-confident winner. The table below is this chapter's own version of that same honesty: which measured forecastability regime a customer falls into, and which of the four baselines above already wins for that regime on this book's own data, not assumed to hold for everyone. Chapter 2 picks up exactly where this table leaves off: whichever regime still leaves real error on the table is where a trained model gets its honest chance to earn its own cost.

In [ ]:
decision = forecastability.join(baseline_results, how="inner")
decision["regime"] = np.select(
    [
        decision["hurst_exponent"] > 0.6,
        decision["permutation_entropy"] < forecastability["permutation_entropy"].median(),
    ],
    ["persistent", "regular"],
    default="volatile / low-persistence",
)


def _dominant_share(group: pd.Series) -> pd.Series:
    counts = group.value_counts()
    return pd.Series({"dominant_baseline": counts.idxmax(), "dominant_share": counts.iloc[0] / len(group) * 100})


regime_summary = (
    decision.groupby("regime")
    .apply(
        lambda g: pd.concat(
            [
                _dominant_share(g["best_baseline"]),
                pd.Series({"customers": len(g), "mean_hurst": g["hurst_exponent"].mean()}),
            ]
        )
    )
    .reset_index()
)
regime_summary["dominant_share"] = regime_summary["dominant_share"].round(0)
regime_summary["mean_hurst"] = regime_summary["mean_hurst"].round(3)
regime_summary = regime_summary[["regime", "customers", "dominant_baseline", "dominant_share", "mean_hurst"]]

themed_gt(
    GT(regime_summary)
    .tab_header(
        title=md("**Which Baseline Wins, By Regime**"),
        subtitle="No single winner dominates every regime the same way",
    )
    .cols_label(
        regime=md("**Forecastability regime**"),
        customers=md("**Customers**"),
        dominant_baseline=md("**Most-often-winning baseline**"),
        dominant_share=md("**Win share (%)**"),
        mean_hurst=md("**Mean Hurst**"),
    )
    .tab_source_note(
        f"n = {len(decision)} real AusNet customers with both a forecastability read and a baseline comparison"
    ),
    n_rows=len(regime_summary),
)

GT(_tbl_data=                       regime  customers dominant_baseline  dominant_share  \
0                  persistent      104.0    window_average            78.0   
1                     regular      126.0    window_average            60.0   
2  volatile / low-persistence      112.0    window_average            73.0   

   mean_hurst  
0       0.651  
1       0.173  
2       0.161  , _body=<great_tables._gt_data.Body object at 0x1329ab680>, _boxhead=Boxhead([ColInfo(var='regime', type=<ColInfoTypeEnum.default: 1>, column_label=Md(text='**Forecastability regime**'), column_align='left', column_width=None), ColInfo(var='customers', type=<ColInfoTypeEnum.default: 1>, column_label=Md(text='**Customers**'), column_align='right', column_width=None), ColInfo(var='dominant_baseline', type=<ColInfoTypeEnum.default: 1>, column_label=Md(text='**Most-often-winning baseline**'), column_align='left', column_width=None), ColInfo(var='dominant_share', type=<ColInfoTypeEnum.default: 1>, column_label=Md(text='**Win share (%)**'), column_align='right', column_width=None), ColInfo(var='mean_hurst', type=<ColInfoTypeEnum.default: 1>, column_label=Md(text='**Mean Hurst**'), column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x132a1b950>, _spanners=Spanners([]), _heading=Heading(title=Md(text='**Which Baseline Wins, By Regime**'), subtitle='No single winner dominates every regime the same way', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x132b9a1e0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x132b9a2d0>, _source_notes=['n = 342 real AusNet customers with both a forecastability read and a baseline comparison'], _footnotes=[], _styles=[StyleInfo(locname=LocTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#171717', font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocSubTitle(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#6B7280', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='regime', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='customers', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='dominant_baseline', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='dominant_share', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='mean_hurst', rownum=None, colnum=None, styles=[CellStyleText(color='#FFFFFF', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocSourceNotes(), grpname=None, colname=None, rownum=None, colnum=None, styles=[CellStyleText(color='#6B7280', font=None, size=None, align=None, v_align=None, style='italic', weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), Sty

In [ ]:
NREL_DIR = Path("../../resources/nrel-buildstock/data")

nrel_meta = pd.read_csv(NREL_DIR / "metadata.csv")
nrel_ts = pd.read_parquet(NREL_DIR / "load_timeseries_30min.parquet")
nrel_ts["building_key"] = nrel_ts["dataset"] + "_" + nrel_ts["bldg_id"].astype(str) + "_" + nrel_ts["state"]
nrel_ts["kw"] = nrel_ts["load_kwh"] * 2.0

# ResStock only: real US single- and multi-family homes, the same broad
# category as AusNet's own residential customers, not ComStock's genuinely
# different-scale commercial buildings.
res_ts = nrel_ts[nrel_ts["dataset"] == "resstock"].sort_values(["building_key", "timestamp"])

series_by_building = {}
for key, grp in res_ts.groupby("building_key"):
    arr = grp["kw"].to_numpy()
    if len(arr) >= len(series_all[0]):
        series_by_building[key] = arr[: len(series_all[0])]

nrel_keys = list(series_by_building.keys())
series_all_nrel = np.stack([series_by_building[k] for k in nrel_keys])
print(f"{len(nrel_keys)} real NREL ResStock buildings with a full real year, shape {series_all_nrel.shape}")

200 real NREL ResStock buildings with a full real year, shape (200, 17520)


In [ ]:
nrel_forecastability = pd.DataFrame([forecastability_row(k, series_by_building[k]) for k in nrel_keys]).set_index(
    "customer_id"
)
nrel_forecastability["spectral_predictability"] = [
    spectral_predictability_score(series_by_building[k]) for k in nrel_keys
]

comparison = pd.DataFrame(
    {
        "AusNet (Australia)": forecastability.mean(),
        "NREL ResStock (US)": nrel_forecastability.mean(),
    }
).round(3)
themed_gt(comparison.reset_index().rename(columns={"index": "Metric"}))

Metric,AusNet (Australia),NREL ResStock (US)
permutation_entropy,0.987,0.939
sample_entropy,0.407,0.394
hurst_exponent,0.315,0.377
dfa_exponent,0.754,0.816
spectral_predictability,0.245,0.364


In [ ]:
nrel_metric_rows = []
for key in nrel_keys:
    x, y = baseline_windows(series_by_building[key])
    actual = y.reshape(-1)
    for name, model in baseline_models.items():
        preds = model.predict(x).reshape(-1)
        row = get_pointwise_metrics(actual, preds, metric_names=METRIC_NAMES)
        row["Model"] = name
        nrel_metric_rows.append(row)

nrel_metrics_all = pd.concat(nrel_metric_rows, ignore_index=True)
nrel_res = nrel_metrics_all.groupby("Model", sort=False)[METRIC_NAMES].mean()
nrel_res.columns = ["MAE", "RMSE", "Corr", "WMAPE", "SMAPE"]
themed_gt(rank_models(nrel_res).rename_axis("Model").reset_index().round(4))

Model,MAE,RMSE,Corr,WMAPE,SMAPE,arith_rank,borda_count,weighted_borda
seasonal_naive,0.4576,0.8419,0.3826,42.8646,34.7729,3.8,19.0,3.7273
window_average,0.4811,0.739,0.2286,45.6263,44.1828,3.2,16.0,3.2727
naive,0.5792,0.9544,0.2001,54.9569,47.0977,2.0,10.0,2.0
drift,0.5953,0.9839,0.1974,56.5744,49.0445,1.0,5.0,1.0


In [ ]:
nrel_win_rows = []
for key in nrel_keys:
    x, y = baseline_windows(series_by_building[key])
    actual = y.reshape(-1)
    maes = {
        name: float(np.mean(np.abs(model.predict(x).reshape(-1) - actual))) for name, model in baseline_models.items()
    }
    nrel_win_rows.append({"building_id": key, "best_baseline": min(maes, key=maes.get)})

nrel_win_tally = (
    pd.DataFrame(nrel_win_rows)["best_baseline"].value_counts(normalize=True).mul(100).round(1).reset_index()
)
nrel_win_tally.columns = ["Best baseline", "Win share (%)"]
themed_gt(nrel_win_tally)

Best baseline,Win share (%)
seasonal_naive,61.0
window_average,36.5
naive,2.5


In [ ]:
regime_baseline_counts = decision.groupby(["regime", "best_baseline"]).size().rename("customers").reset_index()
regime_baseline_counts["share"] = (
    regime_baseline_counts["customers"] / regime_baseline_counts.groupby("regime")["customers"].transform("sum") * 100
)
regime_order = ["persistent", "regular", "volatile / low-persistence"]
regime_baseline_counts["regime"] = pd.Categorical(
    regime_baseline_counts["regime"], categories=regime_order, ordered=True
)

p = (
    ggplot(regime_baseline_counts, aes(x="regime", y="share", fill="best_baseline"))
    + geom_bar(stat="identity", position="stack", width=0.6)
    + scale_fill_manual(values=BASELINE_COLORS)
    + labs(
        title="Which Baseline Wins, By Regime",
        subtitle="Window-average dominates every regime, seasonal-naive is the real runner-up",
        x="",
        y="Share of customers (%)",
        fill="Best baseline",
    )
    + modern_theme()
    + UNBOLD_AXES
    + ggsize(FULL_WIDTH, 320)
)
p